# Transformer-based product understanding

This notebook implements Algorithm 2 from the proposal: use skincare product text and ingredients to create transformer embeddings. These embeddings support product similarity, cold-start recommendations, explanation snippets, and hybrid features for the later Bayesian uncertainty model.

It intentionally uses product metadata only (`product_name`, `brand_name`, categories, highlights, ingredients). It does not use held-out review text for product embeddings, which avoids leaking evaluation labels into the content model.

## 1. Setup

Run this notebook from the `up-skin` folder after activating the local environment:

```bash
source venv/bin/activate
jupyter lab
```

The Kaggle CSVs are expected under `Datasets/`, matching the existing matrix-completion notebooks.

In [ ]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
def find_project_root(start=None):
    """Find the up-skin project root whether the notebook is run from root or a subfolder."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "requirements.txt").exists() and (candidate / "ishita").exists():
            return candidate
    return start

ROOT = find_project_root()
DATA_DIR = ROOT / "Datasets"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

PRODUCT_FILE = DATA_DIR / "product_info.csv"
REVIEW_FILES = [
    DATA_DIR / "reviews_0-250.csv",
    DATA_DIR / "reviews_250-500.csv",
    DATA_DIR / "reviews_500-750.csv",
    DATA_DIR / "reviews_750-1250.csv",
    DATA_DIR / "reviews_1250-end.csv",
]

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
if not PRODUCT_FILE.exists():
    raise FileNotFoundError(
        f"Missing {PRODUCT_FILE}. Add the Kaggle Sephora CSVs under a Datasets/ folder "
        "inside up-skin, then rerun this notebook."
    )

## 2. Load skincare product metadata

The transformer stage works at the product level. Each product gets one clean text document built from stable product metadata and ingredients.

In [ ]:
df_products_raw = pd.read_csv(PRODUCT_FILE)

df_products = df_products_raw.rename(
    columns={
        "rating": "avg_product_rating",
        "reviews": "num_reviews",
    }
)

df_products = df_products[df_products["primary_category"] == "Skincare"].copy()
df_products["product_id"] = df_products["product_id"].astype(str)
df_products = df_products.drop_duplicates(subset=["product_id"]).reset_index(drop=True)

print("Skincare products:", df_products.shape)
df_products[["product_id", "product_name", "brand_name", "secondary_category", "tertiary_category", "ingredients"]].head()

In [ ]:
def parse_listish(value):
    """Convert Kaggle list-like strings such as "['Clean at Sephora']" into readable text."""
    if isinstance(value, list):
        return ", ".join(str(x) for x in value if pd.notna(x))
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return ", ".join(str(x) for x in parsed if pd.notna(x))
        except (SyntaxError, ValueError):
            pass
    return text


def clean_text(value):
    if pd.isna(value):
        return ""
    text = parse_listish(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_product_text(row):
    parts = [
        f"Product: {clean_text(row.get('product_name'))}",
        f"Brand: {clean_text(row.get('brand_name'))}",
        f"Category: {clean_text(row.get('secondary_category'))} {clean_text(row.get('tertiary_category'))}",
        f"Highlights: {clean_text(row.get('highlights'))}",
        f"Ingredients: {clean_text(row.get('ingredients'))}",
    ]
    return " | ".join(part for part in parts if part and not part.endswith(": "))

text_cols = ["product_name", "brand_name", "secondary_category", "tertiary_category", "highlights", "ingredients"]
for col in text_cols:
    df_products[col] = df_products[col].apply(clean_text)

df_products["product_text"] = df_products.apply(build_product_text, axis=1)
df_products["has_ingredients"] = df_products["ingredients"].str.len() > 0

print("Products with ingredients:", int(df_products["has_ingredients"].sum()), "of", len(df_products))
df_products[["product_id", "product_name", "brand_name", "product_text"]].head(3)

## 3. Encode products with a sentence transformer

`all-MiniLM-L6-v2` is small enough for a first pass and gives normalized vectors that work well with cosine similarity.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
product_texts = df_products["product_text"].tolist()

embeddings = model.encode(
    product_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
embeddings = np.asarray(embeddings, dtype=np.float32)

print("Embedding matrix:", embeddings.shape)

In [ ]:
np.savez_compressed(
    ARTIFACT_DIR / "transformer_product_embeddings.npz",
    product_ids=df_products["product_id"].to_numpy(),
    embeddings=embeddings,
)

df_products.drop(columns=["product_text"]).to_csv(ARTIFACT_DIR / "transformer_product_catalog.csv", index=False)
(ARTIFACT_DIR / "product_embedding_model.txt").write_text(MODEL_NAME + "\n")

print("Saved embeddings and product catalog to", ARTIFACT_DIR)

## 4. Product-to-product similarity

This is the core transformer product-understanding output: products are close when their category, benefits, and ingredient text are semantically close.

In [ ]:
product_ids = df_products["product_id"].to_numpy()
product_id_to_idx = {pid: idx for idx, pid in enumerate(product_ids)}

nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(embeddings)


def similar_products(product_id, top_n=10):
    product_id = str(product_id)
    if product_id not in product_id_to_idx:
        raise KeyError(f"Unknown product_id: {product_id}")

    idx = product_id_to_idx[product_id]
    distances, indices = nn.kneighbors(embeddings[idx].reshape(1, -1), n_neighbors=top_n + 1)

    rows = []
    for distance, neighbor_idx in zip(distances[0], indices[0]):
        neighbor_id = product_ids[neighbor_idx]
        if neighbor_id == product_id:
            continue
        item = df_products.iloc[neighbor_idx]
        rows.append({
            "product_id": neighbor_id,
            "similarity": 1 - float(distance),
            "product_name": item["product_name"],
            "brand_name": item["brand_name"],
            "secondary_category": item["secondary_category"],
            "tertiary_category": item["tertiary_category"],
            "price_usd": item.get("price_usd", np.nan),
            "avg_product_rating": item.get("avg_product_rating", np.nan),
        })

    return pd.DataFrame(rows).head(top_n)

sample_product_id = df_products["product_id"].iloc[0]
print("Query product:", sample_product_id, df_products.loc[product_id_to_idx[sample_product_id], "product_name"])
similar_products(sample_product_id, top_n=10)

In [ ]:
def find_products(term, top_n=5):
    mask = df_products["product_text"].str.contains(term, case=False, na=False, regex=False)
    return df_products.loc[mask, ["product_id", "product_name", "brand_name", "secondary_category", "tertiary_category"]].head(top_n)

for term in ["moisturizer", "acne", "sunscreen"]:
    print(f"\nProducts containing '{term}':")
    display(find_products(term, top_n=3))

## 5. Cold-start content recommendations

For a user with only a few liked products, average the liked product embeddings and retrieve nearby products. This directly addresses the matrix model's cold-start weakness because it does not require a learned user latent vector.

In [ ]:
def recommend_from_liked_products(liked_product_ids, top_n=10, min_avg_rating=None):
    liked_product_ids = [str(pid) for pid in liked_product_ids]
    known_likes = [pid for pid in liked_product_ids if pid in product_id_to_idx]
    if not known_likes:
        raise ValueError("None of the liked_product_ids are present in the skincare product catalog.")

    liked_indices = [product_id_to_idx[pid] for pid in known_likes]
    user_vector = embeddings[liked_indices].mean(axis=0)
    norm = np.linalg.norm(user_vector)
    if norm > 0:
        user_vector = user_vector / norm

    scores = embeddings @ user_vector
    candidate = df_products.copy()
    candidate["content_score"] = scores
    candidate = candidate[~candidate["product_id"].isin(known_likes)].copy()

    if min_avg_rating is not None and "avg_product_rating" in candidate:
        candidate = candidate[candidate["avg_product_rating"].fillna(0) >= min_avg_rating]

    cols = [
        "product_id", "content_score", "product_name", "brand_name",
        "secondary_category", "tertiary_category", "price_usd", "avg_product_rating"
    ]
    return candidate.sort_values("content_score", ascending=False)[cols].head(top_n)

liked_example = df_products.head(3)["product_id"].tolist()
print("Liked example product_ids:", liked_example)
recommend_from_liked_products(liked_example, top_n=10)

## 6. Explanation snippets

These explanations are simple and inspectable: they surface shared categories and ingredients between a user's liked products and a recommended product. They are not final clinical claims.

In [ ]:
STOP_INGREDIENTS = {
    "water", "aqua", "eau", "and", "or", "with", "extract", "oil", "acid",
    "sodium", "potassium", "chloride", "alcohol", "fragrance", "parfum",
}


def ingredient_tokens(text):
    text = clean_text(text).lower()
    raw = re.split(r"[,;/()\[\]{}]+", text)
    tokens = []
    for item in raw:
        item = re.sub(r"[^a-z0-9 +.-]", " ", item).strip()
        if not item:
            continue
        words = [w for w in item.split() if len(w) > 2 and w not in STOP_INGREDIENTS]
        if words:
            tokens.append(" ".join(words[:4]))
    return set(tokens)


def explain_recommendation(recommended_product_id, liked_product_ids, max_terms=5):
    recommended_product_id = str(recommended_product_id)
    liked_product_ids = [str(pid) for pid in liked_product_ids if str(pid) in product_id_to_idx]
    if recommended_product_id not in product_id_to_idx or not liked_product_ids:
        return "Not enough product metadata to explain this recommendation."

    rec = df_products.iloc[product_id_to_idx[recommended_product_id]]
    liked = df_products.iloc[[product_id_to_idx[pid] for pid in liked_product_ids]]

    rec_ingredients = ingredient_tokens(rec.get("ingredients", ""))
    liked_ingredients = set().union(*(ingredient_tokens(x) for x in liked["ingredients"].fillna("")))
    shared = sorted(rec_ingredients & liked_ingredients)[:max_terms]

    liked_categories = set(liked["secondary_category"].dropna()) | set(liked["tertiary_category"].dropna())
    rec_categories = {rec.get("secondary_category", ""), rec.get("tertiary_category", "")}
    shared_categories = sorted(x for x in rec_categories & liked_categories if x)[:2]

    reasons = []
    if shared_categories:
        reasons.append("similar category: " + ", ".join(shared_categories))
    if shared:
        reasons.append("shared ingredient signals: " + ", ".join(shared))
    if not reasons:
        reasons.append("its product text is semantically close to products the user liked")

    return "Recommended because " + "; ".join(reasons) + "."

example_recs = recommend_from_liked_products(liked_example, top_n=3)
example_recs["explanation"] = example_recs["product_id"].apply(lambda pid: explain_recommendation(pid, liked_example))
example_recs

## 7. Content evaluation with per-user holdout

This mirrors the matrix-completion split: hold out one product per active user, build each user's content profile from liked training products, and check whether the held-out product appears in the top K recommendations.

In [ ]:
def load_review_interactions():
    missing = [p for p in REVIEW_FILES if not p.exists()]
    if missing:
        print("Skipping interaction evaluation because review CSVs are missing:")
        for p in missing:
            print("-", p)
        return None

    df_reviews = pd.concat((pd.read_csv(p, low_memory=False) for p in REVIEW_FILES), ignore_index=True)
    df_reviews["product_id"] = df_reviews["product_id"].astype(str)

    skincare_ids = set(df_products["product_id"])
    df = df_reviews[df_reviews["product_id"].isin(skincare_ids)][["author_id", "product_id", "rating"]].copy()
    df = df.dropna(subset=["author_id", "product_id", "rating"])
    df["author_id"] = df["author_id"].astype(str)
    df["product_id"] = df["product_id"].astype(str)
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["rating"])
    df = df.groupby(["author_id", "product_id"], as_index=False)["rating"].mean()
    return df

interactions = load_review_interactions()
if interactions is not None:
    print("Interactions:", interactions.shape)
    interactions.head()

In [ ]:
def make_per_user_holdout(df, min_user_ratings=10, min_product_ratings=20, seed=42):
    user_counts = df["author_id"].value_counts()
    product_counts = df["product_id"].value_counts()
    active_users = user_counts[user_counts >= min_user_ratings].index
    active_products = product_counts[product_counts >= min_product_ratings].index
    df_eval = df[df["author_id"].isin(active_users) & df["product_id"].isin(active_products)].copy()

    test_idx = (
        df_eval.groupby("author_id", group_keys=False)
        .apply(lambda x: x.sample(1, random_state=seed))
        .index
    )
    test_df = df_eval.loc[test_idx].copy()
    train_df = df_eval.drop(index=test_idx).copy()

    train_products = set(train_df["product_id"])
    test_df = test_df[test_df["product_id"].isin(train_products)].copy()
    return train_df, test_df


def content_hit_rate_at_k(train_df, test_df, k=10, min_positive_rating=4.0):
    hits = 0
    total = 0
    train_user_items = train_df.groupby("author_id")["product_id"].apply(set).to_dict()

    for user_id, test_rows in test_df.groupby("author_id"):
        true_items = set(test_rows["product_id"])
        user_train = train_df[train_df["author_id"] == user_id]
        liked = user_train[user_train["rating"] >= min_positive_rating]["product_id"].tolist()
        liked = [pid for pid in liked if pid in product_id_to_idx]
        if not liked:
            continue

        liked_indices = [product_id_to_idx[pid] for pid in liked]
        user_vector = embeddings[liked_indices].mean(axis=0)
        norm = np.linalg.norm(user_vector)
        if norm > 0:
            user_vector = user_vector / norm

        scores = embeddings @ user_vector
        already_seen = train_user_items.get(user_id, set())
        ranked_indices = np.argsort(-scores)

        top_items = []
        for idx in ranked_indices:
            pid = product_ids[idx]
            if pid in already_seen:
                continue
            top_items.append(pid)
            if len(top_items) >= k:
                break

        hits += int(bool(true_items & set(top_items)))
        total += 1

    return hits / total if total else np.nan

if interactions is not None:
    train_df, test_df = make_per_user_holdout(interactions)
    print("Train rows:", len(train_df), "Test rows:", len(test_df))
    for k in [5, 10, 20]:
        print(f"Content Hit Rate @{k}: {content_hit_rate_at_k(train_df, test_df, k=k):.4f}")

## 8. Hybrid scoring scaffold

The next notebook can combine these content scores with matrix-factorization scores. Use the matrix notebook's biased MF output as `mf_score`, then blend it with `content_score`.

In [ ]:
def hybrid_score(mf_score, content_score, alpha=0.7):
    """Blend matrix completion and transformer content signals.

    alpha=0.7 keeps matrix completion dominant for known users. Use a lower alpha for sparse or cold-start users.
    """
    return alpha * mf_score + (1 - alpha) * content_score


def build_hybrid_feature_frame(user_id, mf_recommendations, liked_product_ids, alpha=0.7):
    """Combine an MF recommendation table with transformer content scores.

    mf_recommendations must include columns: product_id, mf_score.
    """
    liked_product_ids = [str(pid) for pid in liked_product_ids if str(pid) in product_id_to_idx]
    if not liked_product_ids:
        raise ValueError("Need at least one liked product to compute content scores.")

    liked_indices = [product_id_to_idx[pid] for pid in liked_product_ids]
    user_vector = embeddings[liked_indices].mean(axis=0)
    norm = np.linalg.norm(user_vector)
    if norm > 0:
        user_vector = user_vector / norm

    content_lookup = pd.DataFrame({
        "product_id": product_ids,
        "content_score": embeddings @ user_vector,
    })

    out = mf_recommendations.copy()
    out["product_id"] = out["product_id"].astype(str)
    out = out.merge(content_lookup, on="product_id", how="left")
    out["content_score"] = out["content_score"].fillna(out["content_score"].mean())
    out["hybrid_score"] = hybrid_score(out["mf_score"], out["content_score"], alpha=alpha)
    out["explanation"] = out["product_id"].apply(lambda pid: explain_recommendation(pid, liked_product_ids))
    return out.sort_values("hybrid_score", ascending=False)

# Example shape expected from the matrix-completion notebook:
example_mf_recs = pd.DataFrame({
    "product_id": df_products.head(5)["product_id"].tolist(),
    "mf_score": np.linspace(4.8, 4.2, 5),
})

build_hybrid_feature_frame("example_user", example_mf_recs, liked_example, alpha=0.7)

## What this unlocks next

- Product explanations can now reference ingredient/category similarity.
- Cold-start products can still be retrieved because every product with metadata has an embedding.
- The later Bayesian model can use `mf_score`, `content_score`, user history size, product metadata, and skin-profile fields as input features for final score plus uncertainty.